# Appendix A9-A11: cross-model attribution-score correlation

Reproduces Figures A9 (Automotive), A10 (Career), A11 (Educational) of
`Unequal_influence.pdf` - Spearman correlation between every pair of
models' attribution scores on the same dataset. This is the cheapest item
in the appendix: every model's `attributions.csv` is already under
`results/<dataset>/attributions/` once Figure 5 has run, and correlating them needs no retraining, no GPU, not even a new
evaluation - just reading CSVs that already exist.

**Prerequisite**:

```bash
uv run snakemake figure5
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("../../results")
OUTPUT_DIR = RESULTS / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_LABELS = {"auto": "Automotive", "career": "Career", "edu": "Educational"}

MODEL_LABELS = {
    "olmo": "OLMo 3 7B",
    "qwen2.5-1.5b": "Qwen 2.5 1B",
    "qwen2.5-3b": "Qwen 2.5 3B",
    "qwen2.5-7b": "Qwen 2.5 7B",
    "qwen2.5-14b": "Qwen 2.5 14B",
    "qwen3-4b": "Qwen 3 4B",
    "qwen3-8b": "Qwen 3 8B",
    "qwen3-14b": "Qwen 3 14B",
    "llama3.2-1b": "Llama 3.2 1B",
    "llama3.2-3b": "Llama 3.2 3B",
    "llama3.1-8b": "Llama 3.1 8B",
}
MODEL_ORDER = list(MODEL_LABELS)


## Load every model's attribution scores

Each model ranks the data with cosine similarity against its own baseline.


In [ ]:
def load_model_attributions(dataset, method="cosine"):
    return {path.parent.parent.name: pd.read_csv(path).sort_values("index_example_idx")["attribution"].to_numpy()
            for path in (RESULTS / dataset / "attributions").glob(f"*/{method}/attributions.csv")}


## Figures A9 / A10 / A11

In [ ]:
def plot_attribution_correlation(dataset, model_order=MODEL_ORDER):
    scores = load_model_attributions(dataset)
    models = [model for model in model_order if model in scores]
    corr = pd.DataFrame({model: scores[model] for model in models}).corr(method="spearman").to_numpy()

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    labels = [MODEL_LABELS.get(model, model) for model in models]
    ax.set_xticks(range(len(models))); ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(models))); ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(models)):
        for j in range(len(models)):
            ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax, label="Spearman correlation")
    ax.set_title(f"{DATASET_LABELS.get(dataset, dataset)} attribution-score correlations")
    fig.tight_layout()
    return fig


for dataset in ("auto", "career", "edu"):
    fig = plot_attribution_correlation(dataset)
    fig.savefig(OUTPUT_DIR / f"figure_a9_11_correlation_{dataset}.png", dpi=200, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"figure_a9_11_correlation_{dataset}.pdf", bbox_inches="tight")
    print(f"Saved to {OUTPUT_DIR / f'figure_a9_11_correlation_{dataset}.png'}")
